## Figures settings

In [ ]:
# Import necessary libraries

import numpy as np
from itertools import combinations, product
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from surfplot import Plot
import seaborn as sns
import pandas as pd
import warnings
import variograd_utils as vu
import matplotlib as mpl
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")

train = vu.dataset('train')
test = vu.dataset('test')

In [ ]:
# Set colormaps to use
from variograd_utils import coldhot

k = "#242F36" 
w = "lavender"
c1 = "thistle"
c2 ="lightsteelblue" 
div = sns.color_palette("twilight_shifted", as_cmap=True)
seq = sns.color_palette("PuBuGn_r", as_cmap=True)
qua = sns.color_palette("Set2", as_cmap=True)
bin = LinearSegmentedColormap.from_list("binary", [k, w], N=2)
binc = LinearSegmentedColormap.from_list("binary", [k, c2, "w"], N=255)

# show selected colormaps
sns.palplot(div(np.linspace(0, 1, 9)))
sns.palplot(seq(np.linspace(0, 1, 9)))
sns.palplot(binc(np.linspace(0, 1, 9)))
sns.palplot(bin(np.linspace(0, 1, 9)))
sns.palplot(qua(np.linspace(0, 1, 9)))

In [ ]:
# Define local functions

def style_axis(ax):
    ax.set_facecolor('none')
    ax.set_frame_on(False)
    return ax

def style_figure(figure=None):
    if figure is None:
        figure = plt.gcf()
    figure.set_facecolor('none')
    for ax in figure.get_axes():
        style_axis(ax)

def plot_on_surface(lh_surf, rh_surf, lh_data, rh_data, plot_kwargs=None, layer_kwargs=None):

    plot_kwargs = {} if plot_kwargs is None else plot_kwargs
    layer_kwargs = {} if layer_kwargs is None else layer_kwargs

    data = {"left": vu.left_cortex_data_10k(lh_data, fill=np.nan), "right":vu.right_cortex_data_10k(rh_data, fill=np.nan)}
    p = Plot(lh_surf, rh_surf, **plot_kwargs)
    p.add_layer(data, **layer_kwargs)
    p = vu.Plot_to_numpy(p)
    p[np.all(p==255, axis=2),-1] = 1

    return p

def bootstrap(a, func=None):
    a = np.array(a)
    func = np.mean if func is None else func
    boots = sns.algorithms.bootstrap(a)
    ci = np.percentile(boots, [5, 95])
    stat = func(boots)
    
    return stat, ci

In [ ]:
# Plot settings:

dpi = 600
lmax = 14

plt.rcParams['axes.titlesize'] = 14 # Title size
plt.rcParams['axes.labelsize'] = 12   # Axis labels size
plt.rcParams['xtick.labelsize'] = 10   # X-axis tick labels size
plt.rcParams['ytick.labelsize'] = 10   # Y-axis tick labels size
plt.rcParams['legend.fontsize'] = 12    # Legend font size
plt.rcParams['legend.title_fontsize'] = 12    # Legend title font size
plt.rcParams['axes.facecolor'] = "none"
plt.rcParams['figure.facecolor'] = "none"
plt.rcParams['legend.frameon'] = False
plt.rcParams['axes.edgecolor'] = "none"
plt.rcParams['figure.dpi'] = dpi
plt.rcParams['savefig.dpi'] = dpi
plt.rcParams['figure.constrained_layout.use'] = True


## Figure 1

In [ ]:
# FIGURE 1A
# Plot example gradients on cortex

sidx = [0, 4, 5]
grads = [np.loadtxt(train.outpath(f"train.L.FC_embeddings.a05_t0.G{i}.csv"), delimiter=",")[sidx]
         for i in range(1,4)]

plotsize = (3,2.25)
figsizepx = (int(plotsize[0] * dpi), int(plotsize[1] * dpi))
plot_kwargs = {"layout": "grid", "size": figsizepx, "zoom": 1.6, "views":["lateral"]}
layer_kwargs = {"cmap": "turbo", "zero_transparent": True}
cmax = np.nanmax(np.nanmax(abs(g) for g in grads))

f = plt.figure(figsize=(lmax, lmax/3))
f.set_constrained_layout_pads(w_pad=0.2)

handles = [mpl.patches.Patch(color="k") for i in range(3)]
lgd = f.legend(handles=handles, labels=["001", "002", "N"], title="Participant", ncol=3,
               loc="lower center", bbox_to_anchor=(0.5, 1.))

for i in range(3):
    subj = vu.subject(train.subj_list[sidx[i]], train.id)
    lh_surf = subj.L_midthickness_10k_T1w

    ax_tmp = f.add_subplot(3, 8, i*8+1)
    ax_tmp.axis('off')

    ax_tmp = f.add_subplot(3, 8, i*8+2)
    ax_tmp.axis('off')

    for j in range(3):
        
        lh_data = grads[j][i]
        data = {"left": vu.left_cortex_data_10k(lh_data, fill=np.nan).reshape(-1)}
        p = Plot(surf_lh=lh_surf, **plot_kwargs)
        p.add_layer(data, **layer_kwargs)
        p = vu.Plot_to_numpy(p)
        p[np.all(p==255, axis=2),-1] = 1

        ax = f.add_subplot(3, 8, i*8 + j + 3, aspect=1)
        ax.imshow(p)
        ax.axis('off')
        if i == 0:
            ax.set_title(f" \nGradient {j+1}", pad=10)

    ax_tmp = plt.subplot2grid((3, 8), (0, 5), rowspan=3, colspan=3, projection="3d", aspect="equal")
    ax_tmp.view_init(25,-1)
    ax_tmp.axis('off')



In [ ]:
# FIGURE 1B
# Plot example distance modes on cortex

sidx = [0, 4, 5]
embd = np.load(train.outpath("example_subj_LEM.L.JE_cauchy50.npy"))[sidx, :, :3]

plotsize = (3, 2.25)
figsizepx = (int(plotsize[0] * dpi), int(plotsize[1] * dpi))
plot_kwargs = {"layout": "grid", "size": figsizepx, "zoom": 1.6, "views":["lateral"]}
layer_kwargs = {"cmap": div, "zero_transparent": True}
cmax = np.nanmax(embd)

f = plt.figure(figsize=(lmax, lmax/3))
f.set_constrained_layout_pads(w_pad=0.2)

handles = [mpl.patches.Patch(color="k") for i in range(3)]
lgd = f.legend(handles=handles, labels=["001", "002", "N"], title="Participant", ncol=3,
               loc="lower center", bbox_to_anchor=(0.5, 1.))

for i in range(3):
    subj = vu.subject(train.subj_list[sidx[i]], train.id)
    lh_surf = subj.L_midthickness_10k_T1w
    p = Plot(surf_lh=lh_surf, **plot_kwargs)
    p = vu.Plot_to_numpy(p)
    p[np.all(p==255, axis=2),-1] = 1

    ax = f.add_subplot(3, 8, i*8+1, aspect=1, title=("Individual\nsurfaces" if i==0 else ""))
    ax.imshow(p)
    ax.axis('off')

    ax_tmp = f.add_subplot(3, 8, i*8+2)
    ax_tmp.axis('off')

    for j in range(3):
        
        lh_data = embd[i, :, j]
        data = {"left": vu.left_cortex_data_10k(lh_data, fill=np.nan).reshape(-1)}
        p = Plot(surf_lh=lh_surf, **plot_kwargs)
        p.add_layer(data, **layer_kwargs)
        p = vu.Plot_to_numpy(p)
        p[np.all(p==255, axis=2),-1] = 1

        ax = f.add_subplot(3, 8, i*8 + j + 3, aspect=1)
        ax.imshow(p)
        ax.axis('off')
        if i == 0:
            ax.set_title(f"Component {j+1}", pad=10)

    ax_tmp = plt.subplot2grid((3, 8), (0, 5), rowspan=3, colspan=3, projection="3d", aspect="equal")
    ax_tmp.view_init(25,-1)
    ax_tmp.axis('off')



## Figure 2

### Figure 2A

In [ ]:
# Load errors for all gradients and models
 
scale = 50
metric = "mae"


error_dfs = {}

for grad in [1, 2, 3]:
    fname = train.outpath(f"results/SLresults.{{0}}.G{grad}.S{scale}.side4.centered.{{1}}.h5")

    models = ["Mod_Means", "MeanMod_MeanAvg"]
    hemispheres = ["L", "R"]
    pairs = product(hemispheres, models)
    resdict = {
        (hemi, model, int(sl)): stats for (hemi, model) in pairs
        for sl, stats in vu.load_hdf5(fname.format(hemi, model)).items()
    }

    resdf = pd.DataFrame(resdict).T
    resdf = resdf[[f"{metric}1", f"{metric}2"]]#.apply(np.float32)

    names = ["Means_MeanNull"]
    pairs = product(hemispheres, names)
    for (hemi, model) in pairs:
        res = vu.load_hdf5(fname.format(hemi, model))
        for sl, stats in res.items():
            resdf.loc[(hemi, model, int(sl)), f"{metric}1"] = np.nanmean(stats[f"{metric}1"])
            resdf.loc[(hemi, model, int(sl)), f"{metric}2"] = np.nanmean(stats[f"{metric}2"])

    errors_L = {
        ("L", "baselines + model"): resdf.loc[("L", "Mod_Means"), f"{metric}1"],
        ("L", "baselines"): resdf.loc[("L", "Mod_Means"), f"{metric}2"],
        ("L", "baselines + mean geom."): resdf.loc[("L", "MeanMod_MeanAvg"), f"{metric}2"],
        ("L", "baselines + null geom."): resdf.loc[("L", "Means_MeanNull"), f"{metric}2"]
    }
    errors_L_df = pd.DataFrame(errors_L).T.reset_index(names=["Hemisphere", "Model"])
    errors_L_df = errors_L_df.melt(id_vars=["Hemisphere", "Model"], value_name="Error")

    errors_R = {
        ("R", "baselines + model"): resdf.loc[("R", "Mod_Means"), f"{metric}1"],
        ("R", "baselines"): resdf.loc[("R", "Mod_Means"), f"{metric}2"],
        ("R", "baselines + mean geom."): resdf.loc[("R", "MeanMod_MeanAvg"), f"{metric}2"],
        ("R", "baselines + null geom."): resdf.loc[("R", "Means_MeanNull"), f"{metric}2"],
    }
    errors_R_df = pd.DataFrame(errors_R).T.reset_index(names=["Hemisphere", "Model"])
    errors_R_df = errors_R_df.melt(id_vars=["Hemisphere", "Model"], value_name="Error")

    error_dfs[f"Gradient {grad}"] = pd.concat([errors_L_df, errors_R_df], axis=0).reset_index(drop=True)


In [ ]:
# SUPPLEMENTARY TABLE1 - Statistical tests between models

test = ttest_rel

comparisons = [("baselines + model", "baselines"),
               ("baselines + model", "baselines + mean geom."),
               ("baselines + model", "baselines + null geom."),
               ("baselines + mean geom.", "baselines"),
               ("baselines + null geom.", "baselines")]

combs = list(product(["L", "R"], ["Gradient 1", "Gradient 2", "Gradient 3"], comparisons,
                     [np.nan], [np.nan], [np.nan], [np.nan], [np.nan]))
results = pd.DataFrame(combs, dtype=object,
                       columns=["Hemisphere", "Gradient", "Comparison", "Statistic",
                                "p", "Error difference", "CI", "p_adj"])
results["Comparison"]= results["Comparison"].apply(lambda x: f"{x[0]} vs {x[1]}")
results.set_index(["Hemisphere", "Gradient", "Comparison"], inplace=True)

for grad, errors_LR_df in error_dfs.items():
    left_pvals = pd.DataFrame(columns=errors_LR_df["Model"].unique(), index=errors_LR_df["Model"].unique(), dtype=float)
    left_stats = pd.DataFrame(columns=errors_LR_df["Model"].unique(), index=errors_LR_df["Model"].unique(), dtype=float)

    for hemi in ["L", "R"]:
        for err1, err2 in comparisons:
            err1_df = errors_LR_df[(errors_LR_df["Hemisphere"]==hemi) & (errors_LR_df["Model"]==err1)][["variable","Error"]]
            err2_df = errors_LR_df[(errors_LR_df["Hemisphere"]==hemi) & (errors_LR_df["Model"]==err2)][["variable","Error"]]
            err12_df = err1_df.merge(err2_df, on="variable", suffixes=("1","2"))

            comparison = test(err12_df["Error1"], err12_df["Error2"], nan_policy="omit")
            stat, pval = comparison.statistic, comparison.pvalue
            err_boot, err_ci = bootstrap(err12_df["Error1"] - err12_df["Error2"])
            left_pvals.loc[err1, err2] = pval
            left_stats.loc[err1, err2] = stat
            
            results.loc[(hemi, grad, f"{err1} vs {err2}"), :"CI"] = [f"{stat:.3f}", f"{pval:.0e}", f"{err_boot:.3f}", f"({err_ci[0]:.3f}, {err_ci[1]:.3f})"]
        results.loc[(hemi, grad), "p_adj"] = multipletests(results.loc[(hemi, grad), "p"].astype(float), method="fdr_bh")[1]

results["p_adj"] = results["p_adj"].apply(lambda x: f"{x:.0e}")
results.to_csv(train.outpath(f"results/SL_{metric}_comparisons.centered.s{scale}.csv"))

results

In [ ]:
# FIGURE 2A
# Plot errors with significance markers

f, axs = plt.subplots(1, 6, figsize=(lmax, lmax/3), sharey=True)
colors = qua(np.linspace(0, 1, 8))

for i in range(3):
    df_tmp = error_dfs[f"Gradient {i+1}"].set_index(["Hemisphere", error_dfs[f"Gradient {i+1}"].index])
    axl, axr = axs[i*2], axs[i*2+1]

    for h, ax in zip(["L", "R"], [axl, axr]):
        sns.stripplot(df_tmp.loc[h], x="Model", y="Error", hue="Model", ax=ax, zorder=0,
                      palette=colors, jitter=.3, alpha=.5, s=2, legend=False)
        sns.boxplot(df_tmp.loc[h], x="Model", y="Error", hue="Model", ax=ax, zorder=1,
                    fill=False, linewidth=1, fliersize=0, color=k, gap=0.7, legend=False)
        if (i!=0) | (h=="R"): ax.tick_params(length=0)

        # df_p = results.loc[h, f"Gradient {i+1}"]
        # significant_pairs =  np.array(np.argwhere((df_p < 0.05)))
        ticksidx = {lab.get_text(): i for i, lab in enumerate(ax.get_xticklabels())}
        for j, (pair, p) in enumerate(results.loc[(h, f"Gradient {i+1}"), "p_adj"].items()):
            p = float(p)
            text = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
            x1, x2 = [ticksidx[x] for x in pair.split(" vs ")]
            y = ax.get_ylim()[1] - 0.04 - j * 0.04
            ax.plot([x1, x2], [y, y], color=k, lw=1)
            ax.text((x1 + x2) / 2, y, text, ha="center", va="center", color=k, fontsize=plt.rcParams['xtick.labelsize'])

        ax.set(xlabel=h, xticks=[])

    ax_title = f.add_subplot(1,3, i+1)
    ax_title.set(title=f"Gradient {i+1}")
    ax_title.set_axis_off()

hand_kws = dict(xdata=[0], ydata=[0], lw=0, marker="o", markersize=8)
handles = [mpl.lines.Line2D(**hand_kws, color=color, label=label)
           for color, label in zip(colors[:4], ticksidx.keys())]
f.legend(handles=handles, loc="lower center", ncols=4, frameon=False, bbox_to_anchor=(0.5, 1.0))


### Figure 2B

In [ ]:
# Load statistical results for visualization on cortex

grad = 1
scale = 50
fname = train.outpath(f"results/VTXresults.{{0}}.G{grad}.S{scale}.side4.centered.{{1}}.h5")

x = np.zeros(9398)
lh_stat = {
    "Mod_Means": vu.load_hdf5(fname.format("L", "Mod_Means"))["stat"].squeeze(),
    "MeanMod_MeanAvg": vu.load_hdf5(fname.format("L", "MeanMod_MeanAvg"))["stat"].squeeze(),
    "MeanMod_MeanNull": vu.load_hdf5(fname.format("L", "MeanMod_MeanNull"))["stat"]
}


rh_stat = {
    "Mod_Means": vu.load_hdf5(fname.format("R", "Mod_Means"))["stat"].squeeze(),
    "MeanMod_MeanAvg": vu.load_hdf5(fname.format("R", "MeanMod_MeanAvg"))["stat"].squeeze(),
    "MeanMod_MeanNull": vu.load_hdf5(fname.format("R", "MeanMod_MeanNull"))["stat"].squeeze()
}


a = 0.025
lh_pval = {
    "Mod_Means": vu.load_hdf5(fname.format("L", "Mod_Means"))["pval_adj"].squeeze() >= a,
    "MeanMod_MeanAvg": vu.load_hdf5(fname.format("L", "MeanMod_MeanAvg"))["pval_adj"].squeeze() >= a,
    "MeanMod_MeanNull": vu.load_hdf5(fname.format("L", "MeanMod_MeanNull"))["pval"].squeeze() >= a
}

rh_pval = {
    "Mod_Means": vu.load_hdf5(fname.format("R", "Mod_Means"))["pval_adj"].squeeze() >= a,
    "MeanMod_MeanAvg": vu.load_hdf5(fname.format("R", "MeanMod_MeanAvg"))["pval_adj"].squeeze() >= a,
    "MeanMod_MeanNull": vu.load_hdf5(fname.format("R", "MeanMod_MeanNull"))["pval"].squeeze() >= a
}


In [ ]:
# FIGURE 2B
# Plot gradient statistical results on cortex

scale = 50
comparison = "Mod_Means"
fname = train.outpath(f"results/VTXresults.{{0}}.G{{1}}.S{scale}.side4.centered.{{2}}.h5")

lh_stat = {
    f"Gradient {i}": vu.load_hdf5(fname.format("L", i, comparison))["stat"].squeeze()
    for i in range(1,4)}

rh_stat = {
    f"Gradient {i}": vu.load_hdf5(fname.format("R", i, comparison))["stat"].squeeze()
    for i in range(1,4)}

a = 0.025
lh_pval = {
    f"Gradient {i}": vu.load_hdf5(fname.format("L", i, comparison))["pval_adj"].squeeze() >= a
    for i in range(1,4)}

rh_pval = {
    f"Gradient {i}": vu.load_hdf5(fname.format("R", i, comparison))["pval_adj"].squeeze() >= a
    for i in range(1,4)}


figsize = (lmax,lmax/2.5)

cmax = np.nanmax(np.abs(np.vstack([pd.DataFrame(lh_stat).to_numpy(), pd.DataFrame(rh_stat).to_numpy()])), axis=0).min()
cm1 = div
cm2 = bin.reversed()

lh_surf = train.mesh10k_dir + "/S1200.L.midthickness_MSMAll.10k_fs_LR.surf.gii" 
rh_surf = train.mesh10k_dir + "/S1200.R.midthickness_MSMAll.10k_fs_LR.surf.gii"

plotsize = (6,4.5)
figsizepx = (int(plotsize[0] * dpi), int(plotsize[1] * dpi))
plot_kwargs = {"layout": "grid", "size": figsizepx, "zoom": 1.5, "mirror_views": True}
layer_kwargs = {"cmap": cm1, "color_range": (-cmax, cmax), "zero_transparent": False}

r = 3
grid = (6*r+1, 3)
f = plt.figure(figsize=figsize)

for i in range(3):
    cmax = np.nanmax(np.abs(lh_stat[f"Gradient {i+1}"]))
    layer_kwargs["color_range"] = (-cmax, cmax)
    plot = plot_on_surface(lh_surf, rh_surf, lh_stat[f"Gradient {i+1}"], rh_stat[f"Gradient {i+1}"],
                           plot_kwargs=plot_kwargs,layer_kwargs=layer_kwargs)
    ax = plt.subplot2grid(grid, (0*r, i), rowspan=5*r, colspan=1, title=f"Gradient {i+1}")
    ax.imshow(plot)
    ax.axis('off')



plotsize = (6, 1)
figsizepx = (int(plotsize[0] * dpi), int(plotsize[1] * dpi))
plot_kwargs = {"layout": "row", "size": figsizepx, "zoom": 1.8, "mirror_views": True}
layer_kwargs = {"cmap": cm2, "color_range": (0, 1), "zero_transparent": False}

for i in range(3):
    plot = plot_on_surface(lh_surf, rh_surf, lh_pval[f"Gradient {i+1}"], rh_pval[f"Gradient {i+1}"],
                           plot_kwargs=plot_kwargs,layer_kwargs=layer_kwargs)
    ax = plt.subplot2grid(grid, (5*r, i), rowspan=3*r, colspan=1)
    ax.imshow(plot)
    ax.axis('off')

mappable = plt.cm.ScalarMappable(cmap=cm1, norm=plt.Normalize(vmin=-cmax, vmax=cmax))
cb1 = f.add_subplot(*grid,np.prod(grid)-2)
cb1.set_axis_off()
cb1 = f.colorbar(mappable, ax=[cb1], orientation='horizontal', label='t-value', fraction=1, pad = .5)

mappable = plt.cm.ScalarMappable(cmap=cm2, norm=plt.Normalize(0,1,))
cb2 = f.add_subplot(*grid,np.prod(grid))
cb2.set_axis_off()
cb2 = f.colorbar(mappable, ax=[cb2], orientation='horizontal', label=r"$p_{FDR}$", fraction=1, pad = .5, )
cb2.set_ticks([0.25, 0.75], labels=[fr'$<{a}$',fr'$\geq {a}$'])


### Figure 2C

In [ ]:
# FIGURE 2C
# Correlation between main test and controls

scale = 50
fname = train.outpath(f"results/VTXresults.{{0}}.G{{1}}.S{scale}.side4.centered.{{2}}.h5")

x = np.zeros(9398)
lh_stat = {
    f"Gradient {i}": {
        "Mod_Means": vu.load_hdf5(fname.format("L", i,"Mod_Means"))["stat"].squeeze(),
        "MeanMod_MeanAvg": vu.load_hdf5(fname.format("L", i,"MeanMod_MeanAvg"))["stat"].squeeze(),
        "MeanMod_MeanNull": vu.load_hdf5(fname.format("L", i,"MeanMod_MeanNull"))["stat"]
    }
    for i in range(1,4)
}

rh_stat = {
    f"Gradient {i}": {
        "Mod_Means": vu.load_hdf5(fname.format("R", i,"Mod_Means"))["stat"].squeeze(),
        "MeanMod_MeanAvg": vu.load_hdf5(fname.format("R", i,"MeanMod_MeanAvg"))["stat"].squeeze(),
        "MeanMod_MeanNull": vu.load_hdf5(fname.format("R", i,"MeanMod_MeanNull"))["stat"].squeeze()
    }
    for i in range(1,4)
}

figsize = (lmax,lmax/6)

f, axs = plt.subplots(1, 6, figsize=figsize, dpi=dpi, sharey=True)

scatter_kws = {"s":2, "alpha":.5, "lw":0}

for i, (grad, stat) in enumerate(lh_stat.items()):
    ax1, ax2 = axs[i*2], axs[i*2+1]

    x = np.hstack([stat["Mod_Means"], rh_stat[grad]["Mod_Means"]])
    y_avg = np.hstack([stat["MeanMod_MeanAvg"], rh_stat[grad]["MeanMod_MeanAvg"]])
    y_null = np.hstack([stat["MeanMod_MeanNull"], rh_stat[grad]["MeanMod_MeanNull"]])

    sns.scatterplot(x=x, y=y_avg, ax=ax1, color=c1, **scatter_kws)
    txt = np.min(ax1.get_xlim()) + (np.ptp(ax1.get_xlim()) * 0.1), 0
    ax1.text(*txt, f"R={np.corrcoef(x, y_avg)[0,1]:.2f}", fontsize=plt.rcParams["xtick.labelsize"])

    # ax1.get_yaxis().set_visible(i!=0)
    ax1.set(
        xlabel="t-value\nmean geometry",
        ylabel = ("t-value\nmain test" if i==0 else "")
    )
    ax1.set_xlabel("t-value\nmean geometry")
    ax1.set_xlabel("t-value\nmean geometry")

    sns.scatterplot(y=x, x=y_null, ax=ax2, color=c2, **scatter_kws)
    txt = np.min(ax2.get_xlim()) + (np.ptp(ax2.get_xlim()) * 0.1), 0
    ax2.text(*txt, f"R={np.corrcoef(x, y_null)[0,1]:.2f}", fontsize=plt.rcParams["xtick.labelsize"])
    ax2.set(xlabel="t-value\npermutations")

    ax_title = f.add_subplot(1,3, i+1)
    ax_title.get_yaxis().set_visible(False)
    ax_title.get_xaxis().set_visible(False)
    ax_title.set(title=f"Gradient {i+1}")
    ax_title.set_frame_on(False)
    ax_title.set_ylabel("t-value\nmain test")


## Figure 3

In [ ]:
# Load gradient statistical results and mask by significance

scale = 50
a = 0.025
comparison = "Mod_Means"

fname = train.outpath(f"results/VTXresults.{{0}}.G{{1}}.S{scale}.side4.centered.{{2}}.h5")

gradients = vu.load_hdf5(train.outpath("average_gradients.h5"))

left_pvals = {
    "Gradient 1": vu.load_hdf5(fname.format("L", 1, comparison))["pval_adj"].squeeze(),
    "Gradient 2": vu.load_hdf5(fname.format("L", 2, comparison))["pval_adj"].squeeze(),
    "Gradient 3": vu.load_hdf5(fname.format("L", 3, comparison))["pval_adj"].squeeze()
} 
#vu.load_hdf5(fname.format("L", comparison))["stat"].squeeze()
right_pvals = {
    "Gradient 1": vu.load_hdf5(fname.format("R", 1, comparison))["pval_adj"].squeeze(),
    "Gradient 2": vu.load_hdf5(fname.format("R", 2, comparison))["pval_adj"].squeeze(),
    "Gradient 3": vu.load_hdf5(fname.format("R", 3, comparison))["pval_adj"].squeeze()
} 

left_grads = {
    f"Gradient {i+1}": gradients["L"][:, i].copy() for i in range(gradients["L"].shape[1])
}
right_grads = {
    f"Gradient {i+1}": -gradients["R"][:, i].copy() for i in range(gradients["R"].shape[1])
}

left_grads_clamped = {k: v.copy() for k, v in left_grads.items()}
for key, v in left_grads_clamped.items():
    p = left_pvals[key].copy()
    left_grads_clamped[key][p >= a] = np.nan

right_grads_clamped = {k: v.copy() for k, v in right_grads.items()}
for key, val in right_grads_clamped.items():
    p = right_pvals[key].copy()
    right_grads_clamped[key][p >= a] = np.nan


### Figure 3A

In [ ]:
# FIGURE 3A
# Plot gradients on cortex, masked by significance

figsize = (lmax, 12)
cmap = plt.cm.turbo
cmax = np.nanmax(np.abs(np.vstack([gradients["L"], gradients["R"]])))

lh_surf = train.mesh10k_dir + "/S1200.L.inflated_MSMAll.10k_fs_LR.surf.gii" 
rh_surf = train.mesh10k_dir + "/S1200.R.inflated_MSMAll.10k_fs_LR.surf.gii"

plotsize = (6, 4.5)
figsizepx = (int(plotsize[0] * dpi), int(plotsize[1] * dpi))
print(plotsize, figsizepx)
plot_kwargs = {"layout": "grid", "size": figsizepx, "zoom": 1.5, "mirror_views": True, "brightness": .5}

f = plt.figure(figsize=figsize)
axs13 = []
for i, key in enumerate(left_grads.keys()):

    ax = plt.subplot2grid((4,4), (0,i), rowspan=3, colspan=1)
    axs13.append(ax)
    full_data = {"left": vu.left_cortex_data_10k(left_grads[key], fill=np.nan),
                    "right":vu.right_cortex_data_10k(right_grads[key], fill=np.nan)}
    clamped_data = {"left": vu.left_cortex_data_10k(left_grads_clamped[key], fill=np.nan),
                    "right":vu.right_cortex_data_10k(right_grads_clamped[key], fill=np.nan)}
    white_data = {"left": vu.left_cortex_data_10k(np.ones(9394), fill=np.nan),
                    "right":vu.right_cortex_data_10k(np.ones(9398), fill=np.nan)}

    G_plot = Plot(lh_surf, rh_surf, **plot_kwargs)
    layer_kwargs = {"cmap": "bone", "color_range": (-cmax, cmax)}

    layer_kwargs["cmap"] = cmap
    G_plot.add_layer(clamped_data, **layer_kwargs)

    G_plot = vu.Plot_to_numpy(G_plot)
    G_plot[np.all(G_plot==255, axis=2),-1] = 1

    ax.imshow(G_plot)
    ax.set_title(key)
    ax.set_axis_off()


plotsize = (lmax, 3)
figsizepx = (int(plotsize[0] * dpi), int(plotsize[1] * dpi))
print(plotsize, figsizepx)

ax4 = plt.subplot2grid((4,4), (3,0), rowspan=2, colspan=3)

plot_kwargs = {"size": figsizepx, "zoom": 1.5, "mirror_views": True, "brightness": .7, "layout":"row"}
G_plot = Plot(lh_surf, rh_surf, **plot_kwargs)
cmaps = ["Blues_r", "Greens_r", "Reds_r"]
cmaps = ["blue_transparent", "green_transparent", "red_transparent"]


for i in range(3):
    data_tmp = {"left": vu.left_cortex_data_10k(left_pvals[f"Gradient {i+1}"] < 0.025, fill=np.nan),
                "right":vu.right_cortex_data_10k(right_pvals[f"Gradient {i+1}"] < 0.025, fill=np.nan)}
    G_plot.add_layer(data=data_tmp, cmap=cmaps[i], alpha=0.7, cbar=False)

G_plot = vu.Plot_to_numpy(G_plot)
ax4.imshow(G_plot)
ax4.set_axis_off()


mappable = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=-cmax, vmax=cmax))
f.colorbar(mappable, ax=axs13, location='bottom', label='Functional Gradient', shrink=.3, pad=0.01)

handles = [mpl.patches.Patch(color=plt.cm.get_cmap(cmaps[i])(1), label=f'Gradient {i+1}') for i in range(3)]
ax4.legend(handles=handles, loc='lower center', title=r'$p_{FDR}<0.025$', ncol=3, frameon=False, borderaxespad=-3)


### Figure 3B

In [ ]:
# FIGURE 3B
# Gradient value distributions with significance

a = 0.025
p_df_l = pd.DataFrame.from_dict(left_pvals).set_index([["L"]*9394, np.arange(9394)])
p_df_r = pd.DataFrame.from_dict(right_pvals).set_index([["R"]*9398, np.arange(9398)])
p_df = pd.concat([p_df_l, p_df_r], axis=0)
p_df.columns = ["p1", "p2", "p3"]
p_df["any_significant"] = p_df.min(axis=1) < a
p_df["n_significant"] = (p_df[["p1", "p2", "p3"]] < a).sum(axis=1)
p_df.loc["L", ["G1", "G2", "G3"]] = gradients["L"] 
p_df.loc["R", ["G1", "G2", "G3"]] = gradients["R"] 
p_df[["p1", "p2", "p3"]] = p_df[["p1", "p2", "p3"]] < a
grad_extremes = [("Vis/SM", "DMN"), ("Vis", "SM"), ("DMN", "FPN")]

figsize = (15, 3)
f = plt.figure(figsize=figsize, dpi=dpi, layout="constrained")
histax = [plt.subplot2grid((1,3), (0, i)) for i in range(3)]
twinax = [ax.twinx() for ax in histax]
palette = {True: w, False: k}

hist_args = {"multiple": "fill", "alpha": 1, "lw": 0, "stat": "proportion", "legend": False, "bins": 10}
for i, (ax, ax_second) in enumerate(zip(histax, twinax)):
    i+=1
    sns.histplot(data=p_df, x=f"G{i}", hue=f"p{i}", ax=ax, palette=palette, **hist_args)
    sns.kdeplot(data=p_df, x=f"G{i}", ax=ax_second, fill=False, color=k, cut=0)

    xticks = np.linspace(p_df[f"G{i}"].min(), p_df[f"G{i}"].max(), 10)[[1, -2]]
    ax.set(ylim=(0,1.3), yticks=[0, .5, 1], xticks=xticks, xticklabels=grad_extremes[i-1], xlabel=f"Gradient {i}")
    ax_second.set(ylim=(-5,1.1), yticks=[0,1], ylabel="")


Patch = mpl.patches.Patch
handles = [Patch(facecolor=palette[True], label='p<0.025', alpha=1),
           Patch(color=palette[False], label=r'$p\geq0.025$', alpha=1),
           mpl.lines.Line2D([0], [0], color=palette[False], lw=2, label='Vertex density'),
           ]

f.legend(handles=handles, loc="lower center", ncols=3, frameon=False, markerscale=10, bbox_to_anchor=(0.5, -0.2))


## Figure 4

### Figure 4A,B

In [ ]:
# Load statistical results for scale comparisons

stat = "stat"

fname = train.outpath("results/VTXresults.{0}.G{1}.S{2}.side4.centered.Mod_Means.h5")

lh_stats = {
    f"Gradient {g}": {
        "10": vu.load_hdf5(fname.format("L", g, 10))[stat],
        "50": vu.load_hdf5(fname.format("L", g, 50))[stat],
        "200": vu.load_hdf5(fname.format("L", g, 200))[stat]
        }
    for g in range(1,4)
}

rh_stats = {
    f"Gradient {g}": {
        "10": vu.load_hdf5(fname.format("R", g, 10))[stat],
        "50": vu.load_hdf5(fname.format("R", g, 50))[stat],
        "200": vu.load_hdf5(fname.format("R", g, 200))[stat]
        }
    for g in range(1,4)
}


comparisons = list(combinations(lh_stats["Gradient 1"].keys(), 2))
results = {}
for key, dct in lh_stats.items():
    for (scale1, scale2) in comparisons:
        a = np.hstack([lh_stats[key][scale1], rh_stats[key][scale1]])
        b = np.hstack([lh_stats[key][scale2], rh_stats[key][scale2]])
        tval, pval = ttest_rel(a, b, nan_policy="omit")
        results[(key, scale1, scale2)] = (tval, pval)   


results = pd.DataFrame.from_dict(results).T.rename(columns={0:"Statistic", 1:"p"})
results["p_adj"] = multipletests(results["p"], method="fdr_bh")[1]

t_hm = results.reset_index(names=["Gradient", "Scale 1", "Scale 2"])
t_hm = t_hm.pivot(columns="Gradient", index=["Scale 1", "Scale 2"], values="Statistic")
p_hm = results.reset_index(names=["Gradient", "Scale 1", "Scale 2"])
p_hm = p_hm.pivot(columns="Gradient", index=["Scale 1", "Scale 2"], values="p_adj")

In [ ]:
# Figure 4AB
 
f, axs = plt.subplots(1, 5, figsize=(lmax, lmax/4))

point_kw = {
    "palette": [k],
    "legend" : True,
    "linestyle": "-",
    "markers" : "_",
    "markersize": 20,
    "zorder": 5
    }
plot = sns.pointplot

ref_ax = None
for i, ax in enumerate(axs[:3]):
    df = pd.concat([
    pd.DataFrame(lh_stats[f"Gradient {i+1}"]),
    pd.DataFrame(rh_stats[f"Gradient {i+1}"])
    ]).melt(var_name="scale", value_name="t-value")
    df = df.iloc[0:-1:1]
    c = df.groupby("scale").transform('mean').values.flatten()
    ax.set(xlabel="Kernel scale (mm)", ylabel="t-value", title=f"Gradient {i+1}")
    ax.sharey(axs[0])
    if i != 0:
        ax.get_yaxis().set_visible(False)
    p = plot(df, ax=ax, x="scale", y="t-value", **point_kw)
    ax.axhline(0, color=k, linestyle="--", zorder=0, lw=1)
    sns.stripplot(df, ax=ax, x="scale", y="t-value", color=c2, size=1, alpha=0.1, jitter=0.15)

    for j, scale in enumerate(df.scale.unique()):
        y = df[df.scale==scale]["t-value"].mean()
        ax.text(j, y + 0.1, f"{y:.2f}", ha="center", va="bottom",
                fontsize=plt.rcParams['xtick.labelsize'])



ax_t = axs[3]
sns.heatmap(t_hm, ax=ax_t, cmap=div,
            square=True, annot=True, fmt=".2f", center=True,
            cbar_kws={"shrink": .8, "label": "t-value", "location":"top", "pad":0.01,})
ax_t.set(ylabel="Scale comparison", xlabel="Gradient", xticklabels=[1,2,3])
ax_t.set_xticklabels(labels=ax_t.get_xticklabels(), rotation=0)


ax_p = axs[4]
sns.heatmap(np.log10(p_hm), ax=ax_p, cmap=seq,
            square=True, annot=True, fmt=".0f",
            cbar_kws={"shrink": .8, "label": r"$log_{10}(p)$", "location":"top", "pad":0.01})
ax_p.set(ylabel="Scale comparison", xlabel="Gradient", xticklabels=[1,2,3])
ax_p.set_xticklabels(labels=ax_p.get_xticklabels(), rotation=0)
ax_p.yaxis.set_visible(False)


## Supplementary

### Supplementary  Table 2

In [ ]:
# Load errors for all gradients and models
 
scale = 50
metric = "mae"


error_dfs = {}

for grad in [1, 2, 3]:
    fname = train.outpath(f"results/SLresults.{{0}}.G{grad}.S{scale}.side4.basic.{{1}}.h5")

    models = ["Mod_Avg", "Mod_AvgGeo"]
    hemispheres = ["L", "R"]
    pairs = product(hemispheres, models)
    resdict = {
        (hemi, model, int(sl)): stats for (hemi, model) in pairs
        for sl, stats in vu.load_hdf5(fname.format(hemi, model)).items()
    }

    resdf = pd.DataFrame(resdict).T
    resdf = resdf[[f"{metric}1", f"{metric}2"]]#.apply(np.float32)

    names = ["Mod_NullGeo"]
    pairs = product(hemispheres, names)
    for (hemi, model) in pairs:
        res = vu.load_hdf5(fname.format(hemi, model))
        for sl, stats in res.items():
            resdf.loc[(hemi, model, int(sl)), f"{metric}1"] = np.nanmean(stats[f"{metric}1"])
            resdf.loc[(hemi, model, int(sl)), f"{metric}2"] = np.nanmean(stats[f"{metric}2"])

    errors_L = {
        ("L", "model"): resdf.loc[("L", "Mod_Avg"), f"{metric}1"],
        ("L", "average"): resdf.loc[("L", "Mod_Avg"), f"{metric}2"],
        ("L", "mean geom."): resdf.loc[("L", "Mod_AvgGeo"), f"{metric}2"],
        ("L", "null geom."): resdf.loc[("L", "Mod_NullGeo"), f"{metric}2"]
    }
    errors_L_df = pd.DataFrame(errors_L).T.reset_index(names=["Hemisphere", "Model"])
    errors_L_df = errors_L_df.melt(id_vars=["Hemisphere", "Model"], value_name="Error")

    errors_R = {
        ("R", "model"): resdf.loc[("R", "Mod_Avg"), f"{metric}1"],
        ("R", "average"): resdf.loc[("R", "Mod_Avg"), f"{metric}2"],
        ("R", "mean geom."): resdf.loc[("R", "Mod_AvgGeo"), f"{metric}2"],
        ("R", "null geom."): resdf.loc[("R", "Mod_NullGeo"), f"{metric}2"]
    }

    errors_R_df = pd.DataFrame(errors_R).T.reset_index(names=["Hemisphere", "Model"])
    errors_R_df = errors_R_df.melt(id_vars=["Hemisphere", "Model"], value_name="Error")

    error_dfs[f"Gradient {grad}"] = pd.concat([errors_L_df, errors_R_df], axis=0).reset_index(drop=True)


In [ ]:
# Statistical tests between models

test = ttest_rel

comparisons = [("model", "average"),
               ("model", "mean geom."),
               ("model", "null geom.")]

combs = list(product(["L", "R"], ["Gradient 1", "Gradient 2", "Gradient 3"], comparisons,
                     [np.nan], [np.nan], [np.nan], [np.nan], [np.nan]))
results = pd.DataFrame(combs, dtype=object,
                       columns=["Hemisphere", "Gradient", "Comparison", "Statistic",
                                "p", "Error difference", "CI", "p_adj"])
results["Comparison"]= results["Comparison"].apply(lambda x: f"{x[0]} vs {x[1]}")
results.set_index(["Hemisphere", "Gradient", "Comparison"], inplace=True)

for grad, errors_LR_df in error_dfs.items():
    left_pvals = pd.DataFrame(columns=errors_LR_df["Model"].unique(), index=errors_LR_df["Model"].unique(), dtype=float)
    left_stats = pd.DataFrame(columns=errors_LR_df["Model"].unique(), index=errors_LR_df["Model"].unique(), dtype=float)

    for hemi in ["L", "R"]:
        for err1, err2 in comparisons:
            err1_df = errors_LR_df[(errors_LR_df["Hemisphere"]==hemi) & (errors_LR_df["Model"]==err1)][["variable","Error"]]
            err2_df = errors_LR_df[(errors_LR_df["Hemisphere"]==hemi) & (errors_LR_df["Model"]==err2)][["variable","Error"]]
            err12_df = err1_df.merge(err2_df, on="variable", suffixes=("1","2"))

            comparison = test(err12_df["Error1"], err12_df["Error2"], nan_policy="omit")
            stat, pval = comparison.statistic, comparison.pvalue
            err_boot, err_ci = bootstrap(err12_df["Error1"] - err12_df["Error2"])
            left_pvals.loc[err1, err2] = pval
            left_stats.loc[err1, err2] = stat
            
            results.loc[(hemi, grad, f"{err1} vs {err2}"), :"CI"] = [f"{stat:.3f}", f"{pval:.0e}", f"{err_boot:.3f}", f"({err_ci[0]:.3f}, {err_ci[1]:.3f})"]
        results.loc[(hemi, grad), "p_adj"] = multipletests(results.loc[(hemi, grad), "p"].astype(float), method="fdr_bh")[1]

results["p_adj"] = results["p_adj"].apply(lambda x: f"{x:.0e}")
results.to_csv(train.outpath(f"results/SL_{metric}_comparisons.uncentered.s{scale}.csv"))

results

### Supplementary Figure 1

In [ ]:
# FIGURE S1A
# Plot errors with significance markers

f, axs = plt.subplots(1, 6, figsize=(lmax, lmax/3), sharey=True)
colors = qua(np.linspace(0, 1, 8))

for i in range(3):
    df_tmp = error_dfs[f"Gradient {i+1}"].set_index(["Hemisphere", error_dfs[f"Gradient {i+1}"].index])
    axl, axr = axs[i*2], axs[i*2+1]

    for h, ax in zip(["L", "R"], [axl, axr]):
        sns.stripplot(df_tmp.loc[h], x="Model", y="Error", hue="Model", ax=ax, zorder=0,
                      palette=colors, jitter=.3, alpha=.5, s=2, legend=False)
        sns.boxplot(df_tmp.loc[h], x="Model", y="Error", hue="Model", ax=ax, zorder=1,
                    fill=False, linewidth=1, fliersize=0, color=k, gap=0.7, legend=False)
        if (i!=0) | (h=="R"): ax.tick_params(length=0)

        ticksidx = {lab.get_text(): i for i, lab in enumerate(ax.get_xticklabels())}
        for j, (pair, p) in enumerate(results.loc[(h, f"Gradient {i+1}"), "p_adj"].items()):
            p = float(p)
            text = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
            x1, x2 = [ticksidx[x] for x in pair.split(" vs ")]
            y = ax.get_ylim()[1] - 0.04 - j * 0.04
            ax.plot([x1, x2], [y, y], color=k, lw=1)
            ax.text((x1 + x2) / 2, y, text, ha="center", va="center", color=k, fontsize=plt.rcParams['xtick.labelsize'])

        ax.set(xlabel=h, xticks=[])

    ax_title = f.add_subplot(1,3, i+1, sharey=ax)
    ax_title.set(title=f"Gradient {i+1}")
    ax_title.set_axis_off()

hand_kws = dict(xdata=[0], ydata=[0], lw=0, marker="o", markersize=8)
handles = [mpl.lines.Line2D(**hand_kws, color=color, label=label)
           for color, label in zip(colors[:4], ticksidx.keys())]
f.legend(handles=handles, loc="lower center", ncols=4, frameon=False, bbox_to_anchor=(0.5, 1.0))
